# ThermHand final pipeline

Run in order: **1) install**, **2) pipeline** (extracts features and fits the cohort model), **3) export held-out bundle** (writes `model_bundle_excl_S17.json`).

If you change any mask or feature code, delete `features_cache.pkl` or re-run cell 2 without `--reuse-features`.

In [1]:
!pip install -q mediapipe


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.


In [2]:
"""thermhand FINAL (v6 + hysteresis mask): subspace + residual scorer, ablations, label-shuffle control, LOSO validation."""
import argparse
import json
import pickle
import re
import time
import urllib.request
import warnings
import zlib
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import spearmanr
from sklearn.covariance import EmpiricalCovariance, MinCovDet
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    import mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision as mp_vision
    HAS_MEDIAPIPE = True
except Exception as _e:
    HAS_MEDIAPIPE = False
    print(f"[WARN] MediaPipe import failed: {_e!r}", flush=True)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torchvision.models as tv_models
    HAS_TORCH = True
except Exception:
    HAS_TORCH = False

DATA_ROOT = "/kaggle/input/datasets/podakantisatyajith/thermal-data-collection/FLIROne"
WORK_ROOT = "/kaggle/working/thermhand_out/final_v6"
SEG_CKPT = "/kaggle/working/thermhand_out/m1_seg/m1_final.pt"
TASK_PATH = "/tmp/hand_landmarker.task"
TASK_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
DISCLAIMER = "relative to the 17-subject cohort, not a diagnosis"

SEG_INPUT_HW = (120, 160)
DS = 4
FOCAL_DELTAS = (0.5, 1.0, 2.0)
REGIONAL_DELTAS = (1.0, 2.0, 3.0)
ORIENT_CANDIDATES = ["rot_ccw", "rot_cw", "T", "T_rot180"]
FINGER_CHAINS = {
    "thumb": [1, 2, 3, 4],
    "index": [5, 6, 7, 8],
    "middle": [9, 10, 11, 12],
    "ring": [13, 14, 15, 16],
    "pinky": [17, 18, 19, 20],
}
PALM_IDXS = [0, 5, 9, 13, 17]
LR_BASE = ["f_mask_mean_c", "f_focal_peak_c", "f_local_joint_contrast_c", "f_joint_p90_contrast_c", "f_palm_minus_tip_c"]

GATE_MIN_MASK_OK_FRAC = 0.85
GATE_MAX_AREA_CV = 0.35
GATE_MIN_VALID_JOINT_FRAC = 0.50
MIN_FIT_RECORDINGS = 8


# ----------------------------------------------------------------------------- utils
def _jsonable(o: Any) -> Any:
    if isinstance(o, dict):
        return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return _jsonable(o.tolist())
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, (int, np.integer)):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return float(o) if np.isfinite(o) else None
    return o


def parse_recording_meta(p: Path, idx: int) -> tuple[str, str, str]:
    subj = None
    for tok in reversed(p.parts):
        m = re.fullmatch(r"[sS](\d+)", tok.strip())
        if m:
            subj = f"S{int(m.group(1))}"
            break
    if subj is None:
        subj = f"S{(idx // 4) + 1}"
    low = str(p).lower()
    hand = "left" if "left" in low else "right" if "right" in low else ("left" if idx % 2 == 0 else "right")
    stem = p.stem.lower()
    task = "open_close" if ("open" in stem or "close" in stem or "open" in low or "close" in low) else "static"
    return subj, hand, task


def find_rgb_for_npy(subj: str, hand: str, task: str, all_mp4: list[Path]) -> Path | None:
    cands = [
        p for p in all_mp4
        if any(t.lower() == subj.lower() for t in p.parts)
        and "thermal" not in p.name.lower()
        and "radiometric" not in p.name.lower()
    ]
    task_toks = ["open", "close"] if task == "open_close" else ["static"]
    scored = []
    for p in cands:
        s = str(p).lower()
        sc = (2 if hand in s else 0) + (2 if any(t in s for t in task_toks) else 0) + (1 if "rgb" in s else 0)
        scored.append((sc, p))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1] if scored and scored[0][0] >= 2 else None


def to_celsius(raw: np.ndarray) -> np.ndarray:
    arr = np.asarray(raw, dtype=np.float32)
    if arr.ndim == 3:
        arr = arr[..., 0]
    med = float(np.nanmedian(arr))
    if med > 10000.0:
        return arr / 100.0 - 273.15
    if med > 180.0:
        return arr - 273.15
    return arr


def orient_thermal(arr: np.ndarray, mode: str) -> np.ndarray:
    a = np.asarray(arr, dtype=np.float32)
    if a.shape == (480, 640):
        out = a
    elif mode == "T":
        out = a.T
    elif mode == "rot_ccw":
        out = np.rot90(a, 1)
    elif mode == "rot_cw":
        out = np.rot90(a, -1)
    elif mode == "T_rot180":
        out = a.T[::-1, ::-1]
    else:
        raise ValueError(mode)
    if out.shape != (480, 640):
        out = cv2.resize(np.ascontiguousarray(out), (640, 480), interpolation=cv2.INTER_LINEAR)
    out = np.ascontiguousarray(out, dtype=np.float32)
    bad = ~np.isfinite(out)
    if bad.any():
        fill = float(np.nanmedian(out)) if (~bad).any() else 30.0
        out = np.where(bad, fill, out).astype(np.float32)
    return out


def canon_rgb(rgb: np.ndarray, k: int) -> np.ndarray:
    a = np.rot90(rgb, k) if k else rgb
    if a.shape[:2] != (480, 640):
        a = cv2.resize(np.ascontiguousarray(a), (640, 480), interpolation=cv2.INTER_AREA)
    return np.ascontiguousarray(a, dtype=np.uint8)


def read_rgb(cap: cv2.VideoCapture, ri: int) -> np.ndarray | None:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(ri))
    ok, bgr = cap.read()
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB) if ok and bgr is not None else None


# ----------------------------------------------------------------------------- masks
def fill_holes(m: np.ndarray) -> np.ndarray:
    """Fill background pockets fully enclosed by the hand mask (gaps between fingers and palm)."""
    inv = (1 - m).astype(np.uint8)
    if inv[0, 0] == 1:
        ff = np.zeros((inv.shape[0] + 2, inv.shape[1] + 2), np.uint8)
        cv2.floodFill(inv, ff, (0, 0), 2)
        holes = inv == 1
    else:
        holes = np.zeros(m.shape, dtype=bool)
    return (m.astype(bool) | holes).astype(np.uint8)


def otsu_mask(th_c: np.ndarray) -> np.ndarray:
    """Otsu + hysteresis. Plain Otsu dropped cooler fingertips; weak regions touching a strong
    hand pixel are kept, isolated background noise is not."""
    bg, hi = float(np.percentile(th_c, 8)), float(np.percentile(th_c, 97))
    span = max(0.5, hi - bg)
    u8 = (np.clip((th_c - bg) / span, 0.0, 1.0) * 255.0).astype(np.uint8)
    t, _ = cv2.threshold(u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    strong = u8 > t
    weak = u8 > max(20, int(0.55 * t))
    n, lbl = cv2.connectedComponents(weak.astype(np.uint8), connectivity=8)
    if n <= 1:
        return np.zeros(u8.shape, dtype=np.uint8)
    keep = np.unique(lbl[strong])
    keep = keep[keep > 0]
    if keep.size == 0:
        return np.zeros(u8.shape, dtype=np.uint8)
    m = np.isin(lbl, keep).astype(np.uint8)
    n2, lbl2, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n2 > 1:
        m = (lbl2 == 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    return fill_holes(m)


def mask_ok(m: np.ndarray) -> bool:
    return 0.03 <= float(m.mean()) <= 0.60


if HAS_TORCH:
    class TransferSegUNet(nn.Module):
        def __init__(self, in_channels: int = 2) -> None:
            super().__init__()
            bb = tv_models.mobilenet_v3_small(weights=None)
            fc = bb.features[0][0]
            bb.features[0][0] = nn.Conv2d(
                in_channels, fc.out_channels, fc.kernel_size, fc.stride, fc.padding, bias=fc.bias is not None
            )
            self.enc_low = nn.Sequential(bb.features[0], bb.features[1])
            self.enc_mid = nn.Sequential(bb.features[2], bb.features[3])
            self.enc_high = nn.Sequential(*list(bb.features[4:]))
            self.up1 = nn.ConvTranspose2d(576, 48, 2, 2)
            self.dec1 = nn.Sequential(nn.Conv2d(72, 48, 3, padding=1), nn.BatchNorm2d(48), nn.SiLU())
            self.up2 = nn.ConvTranspose2d(48, 24, 2, 2)
            self.dec2 = nn.Sequential(nn.Conv2d(40, 24, 3, padding=1), nn.BatchNorm2d(24), nn.SiLU())
            self.head = nn.Conv2d(24, 1, 1)

        def forward(self, x):
            h, w = x.shape[-2:]
            f1 = self.enc_low(x)
            f2 = self.enc_mid(f1)
            f3 = self.enc_high(f2)
            u1 = F.interpolate(self.up1(f3), size=f2.shape[-2:], mode="bilinear", align_corners=False)
            d1 = self.dec1(torch.cat([u1, f2], 1))
            u2 = F.interpolate(self.up2(d1), size=f1.shape[-2:], mode="bilinear", align_corners=False)
            d2 = self.dec2(torch.cat([u2, f1], 1))
            return F.interpolate(self.head(d2), size=(h, w), mode="bilinear", align_corners=False)


def load_seg_model(ckpt: Path, device):
    if not (HAS_TORCH and ckpt.exists()):
        return None
    m = TransferSegUNet(2).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    return m.eval()


def unet_mask(th_c: np.ndarray, model, device) -> np.ndarray:
    small = cv2.resize(th_c, (SEG_INPUT_HW[1], SEG_INPUT_HW[0]), interpolation=cv2.INTER_AREA)
    lo, hi = float(np.percentile(small, 2)), float(np.percentile(small, 98))
    ch0 = np.clip((small - lo) / max(1e-5, hi - lo), 0, 1)
    ch1 = np.clip((small - 15.0) / 25.0, 0, 1)
    with torch.no_grad():
        inp = torch.from_numpy(np.stack([ch0, ch1])[None].astype(np.float32)).to(device)
        prob = torch.sigmoid(model(inp))[0, 0].cpu().numpy()
    prob = cv2.resize(prob, (640, 480), interpolation=cv2.INTER_LINEAR)
    return (prob >= 0.5).astype(np.uint8)


def get_mask(th_c, source: str, model, device) -> tuple[np.ndarray, str]:
    otsu = otsu_mask(th_c)
    if source == "otsu" or model is None:
        return otsu, "otsu"
    m = unet_mask(th_c, model, device)
    return (m, "unet") if mask_ok(m) else (otsu, "otsu_fallback")


# ----------------------------------------------------------------------------- landmarks & registration
def ensure_task(path: Path) -> bool:
    if path.is_file() and path.stat().st_size > 100_000:
        return True
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(TASK_URL, str(path))
        return path.is_file() and path.stat().st_size > 100_000
    except Exception as e:
        print(f"[WARN] could not fetch hand_landmarker.task: {e!r}", flush=True)
        return False


def make_detector(task_path: Path):
    if not HAS_MEDIAPIPE or not ensure_task(task_path):
        return None
    opts = mp_vision.HandLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=str(task_path)),
        running_mode=mp_vision.RunningMode.IMAGE,
        num_hands=1,
        min_hand_detection_confidence=0.3,
        min_hand_presence_confidence=0.3,
        min_tracking_confidence=0.3,
    )
    return mp_vision.HandLandmarker.create_from_options(opts)


def detect_pts(det, rgb_c: np.ndarray) -> np.ndarray | None:
    res = det.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_c))
    if not res.hand_landmarks:
        return None
    return np.array([[p.x * 640.0, p.y * 480.0] for p in res.hand_landmarks[0]], dtype=np.float32)


def choose_rgb_rotation(det, npy_files, mp4_files, meta, n_rec=10, n_frames=4):
    if det is None:
        return 0, {}
    hits, tries = {0: 0, 1: 0, 2: 0, 3: 0}, 0
    step = max(1, len(npy_files) // n_rec)
    for idx in list(range(0, len(npy_files), step))[:n_rec]:
        subj, hand, task = meta[idx]
        rgb_p = find_rgb_for_npy(subj, hand, task, mp4_files)
        if rgb_p is None:
            continue
        cap = cv2.VideoCapture(str(rgb_p))
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        for ri in np.linspace(0, max(0, n - 1), n_frames).astype(int):
            rgb = read_rgb(cap, ri)
            if rgb is None:
                continue
            tries += 1
            for k in hits:
                hits[k] += int(detect_pts(det, canon_rgb(rgb, k)) is not None)
        cap.release()
    rates = {k: v / max(1, tries) for k, v in hits.items()}
    order = [0, 1, 3, 2]
    return max(order, key=lambda k: (rates[k], -order.index(k))), rates


def ds_mask(m: np.ndarray) -> np.ndarray:
    return (cv2.resize(m.astype(np.float32), (640 // DS, 480 // DS), interpolation=cv2.INTER_AREA) >= 0.5).astype(np.uint8)


def hull_mask(pts: np.ndarray, grow: float = 0.06) -> np.ndarray:
    m = np.zeros((480, 640), np.uint8)
    cv2.fillConvexPoly(m, cv2.convexHull(np.round(pts).astype(np.int32)), 1)
    r = int(max(2, grow * np.sqrt(max(1, int(m.sum())))))
    return cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1)))


def make_M(s, ang, tx, ty, cx=320.0, cy=240.0) -> np.ndarray:
    c, sn = np.cos(ang), np.sin(ang)
    A = s * np.array([[c, -sn], [sn, c]])
    t = np.array([cx, cy]) - A @ np.array([cx, cy]) + np.array([tx, ty])
    return np.hstack([A, t[:, None]]).astype(np.float32)


def ds_M(M: np.ndarray) -> np.ndarray:
    M2 = M.astype(np.float32).copy()
    M2[:, 2] /= DS
    return M2


def warp_iou(rgb_ds, th_ds, M) -> float:
    w = cv2.warpAffine(rgb_ds, ds_M(M), (640 // DS, 480 // DS), flags=cv2.INTER_NEAREST)
    inter = float(np.logical_and(w > 0, th_ds > 0).sum())
    union = float(np.logical_or(w > 0, th_ds > 0).sum())
    return inter / union if union else 0.0


def pts_ds_xy(pts_ds, M) -> tuple[np.ndarray, np.ndarray]:
    q = cv2.transform(pts_ds.reshape(-1, 1, 2).astype(np.float32), ds_M(M)).reshape(-1, 2)
    xi = np.clip(np.round(q[:, 0]).astype(int), 0, 640 // DS - 1)
    yi = np.clip(np.round(q[:, 1]).astype(int), 0, 480 // DS - 1)
    return xi, yi


def collect_pairs(npy_files, mp4_files, meta, det, rgb_k, orients, fracs=(0.25, 0.5, 0.75)):
    pairs, tried, hit = [], 0, 0
    for idx, np_p in enumerate(npy_files):
        subj, hand, task = meta[idx]
        rgb_p = find_rgb_for_npy(subj, hand, task, mp4_files)
        if rgb_p is None:
            continue
        th_arr = np.load(np_p, mmap_mode="r")
        n_th = len(th_arr)
        cap = cv2.VideoCapture(str(rgb_p))
        if not cap.isOpened():
            continue
        n_rgb = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        for fr in fracs:
            ti = int(round(fr * (n_th - 1)))
            ri = int(round(fr * max(0, n_rgb - 1)))
            rgb = read_rgb(cap, ri)
            if rgb is None:
                continue
            tried += 1
            pts = detect_pts(det, canon_rgb(rgb, rgb_k))
            if pts is None:
                continue
            hit += 1
            th0 = to_celsius(th_arr[ti])
            th_ds = {}
            for o in orients:
                m = otsu_mask(orient_thermal(th0, o))
                if mask_ok(m):
                    th_ds[o] = ds_mask(m)
            if th_ds:
                pairs.append({"subject": subj, "pts_ds": pts / DS, "rgb_ds": ds_mask(hull_mask(pts)), "th_ds": th_ds})
        cap.release()
    return pairs, hit / max(1, tried)


def palm_center_radius(m: np.ndarray):
    dt = cv2.distanceTransform(m.astype(np.uint8), cv2.DIST_L2, 5)
    _, mx, _, loc = cv2.minMaxLoc(dt)
    return np.array(loc, np.float64), float(mx)


def fit_affine(pairs, orient):
    use = []
    for p in pairs:
        th = p["th_ds"].get(orient)
        if th is None or th.sum() < 20 or p["rgb_ds"].sum() < 20:
            continue
        dist = cv2.distanceTransform((1 - th).astype(np.uint8), cv2.DIST_L2, 3)
        use.append((p["rgb_ds"], th, dist, p["pts_ds"]))
    if len(use) < 5:
        return None, np.inf

    # Palm-anchored start: scale from palm inscribed radii, translation from palm centres.
    cr_l, ct_l, s_l = [], [], []
    for rgb_ds, th_ds, _, _ in use:
        cr, rr = palm_center_radius(rgb_ds)
        ct, rt = palm_center_radius(th_ds)
        if rr >= 2 and rt >= 2:
            cr_l.append(cr * DS)
            ct_l.append(ct * DS)
            s_l.append(rt / rr)
    if len(s_l) < 5:
        return None, np.inf
    cr_m, ct_m = np.median(cr_l, axis=0), np.median(ct_l, axis=0)
    s_pal = float(np.median(s_l))

    def obj(x):
        M = make_M(*x)
        vals = []
        for rgb_ds, th_ds, dist, pd_ in use:
            xi, yi = pts_ds_xy(pd_, M)
            vals.append(0.5 * warp_iou(rgb_ds, th_ds, M) + 0.5 * float(np.mean(np.exp(-dist[yi, xi] / 2.0))))
        return -float(np.mean(vals))

    c0 = np.array([320.0, 240.0])
    best = None
    for mult in (0.9, 1.0, 1.1):
        s0 = s_pal * mult
        t0 = ct_m - c0 - s0 * (cr_m - c0)
        x0 = np.array([s0, 0.0, t0[0], t0[1]])
        simplex = np.vstack([x0] + [x0 + np.eye(4)[i] * np.array([0.02, 0.01, 4.0, 4.0])[i] for i in range(4)])
        res = minimize(obj, x0, method="Nelder-Mead",
                       options={"initial_simplex": simplex, "maxiter": 300, "xatol": 1e-3, "fatol": 1e-4})
        if best is None or res.fun < best.fun:
            best = res
    return make_M(*best.x), float(best.fun)


def eval_registration(pairs, M, orient) -> dict:
    ious, hit, tot = [], 0, 0
    for p in pairs:
        th = p["th_ds"].get(orient)
        if th is None:
            continue
        ious.append(warp_iou(p["rgb_ds"], th, M))
        dil = cv2.dilate(th, np.ones((3, 3), np.uint8))
        xi, yi = pts_ds_xy(p["pts_ds"], M)
        hit += int(dil[yi, xi].sum())
        tot += 21
    return {
        "n_pairs": len(ious),
        "median_iou": float(np.median(ious)) if ious else np.nan,
        "joint_in_mask_tol_pct": 100.0 * hit / max(1, tot),
    }


def fit_registration(npy_files, mp4_files, meta, det, rgb_k, seed) -> dict:
    first = np.load(npy_files[0], mmap_mode="r")[0]
    orients = ["T"] if np.asarray(first).shape[:2] == (480, 640) else ORIENT_CANDIDATES
    pairs, hit_rate = collect_pairs(npy_files, mp4_files, meta, det, rgb_k, orients)
    out = {
        "M": make_M(1.97, 0, -211.0, -296.0),
        "orient": orients[0],
        "joints_ok": False,
        "mp_hit_rate_calibration": hit_rate,
        "n_pairs": len(pairs),
    }
    if len(pairs) < 12:
        return out
    subs = sorted({p["subject"] for p in pairs})
    perm = list(np.random.default_rng(seed).permutation(subs))
    calib_s = set(perm[: max(1, int(round(0.6 * len(subs))))])
    calib = [p for p in pairs if p["subject"] in calib_s]
    held = [p for p in pairs if p["subject"] not in calib_s] or calib
    per_orient, best = {}, None
    for o in orients:
        M, fun = fit_affine(calib, o)
        if M is None:
            continue
        per_orient[o] = {"fit_score": -fun, **{f"heldout_{k}": v for k, v in eval_registration(held, M, o).items()}}
        if best is None or fun < best[2]:
            best = (o, M, fun)
    if best is None:
        return out
    o, M_cal, _ = best
    ev = eval_registration(held, M_cal, o)
    M_final, _ = fit_affine(pairs, o)
    ok = ev["joint_in_mask_tol_pct"] >= 85.0 and ev["median_iou"] >= 0.5
    out.update({
        "M": M_final if M_final is not None else M_cal,
        "orient": o,
        "joints_ok": bool(ok),
        "heldout": ev,
        "per_orient": per_orient,
        "heldout_is_separate_subjects": held is not calib,
    })
    print(
        f"[REG] orient={o} heldout IoU={ev['median_iou']:.3f} joints-in-mask(tol)={ev['joint_in_mask_tol_pct']:.1f}% "
        f"-> joints {'ENABLED' if ok else 'DISABLED (gate failed)'}",
        flush=True,
    )
    return out


# ----------------------------------------------------------------------------- features
def focal_peak_c(th_c, th_m, scale) -> float:
    t = cv2.resize(th_c, (320, 240), interpolation=cv2.INTER_AREA)
    m = cv2.resize(th_m.astype(np.float32), (320, 240), interpolation=cv2.INTER_NEAREST)
    s1, s2 = max(1.5, 0.02 * scale / 2.0), max(6.0, 0.10 * scale / 2.0)
    tm = t * m

    def nb(sig):
        return cv2.GaussianBlur(tm, (0, 0), sig) / (cv2.GaussianBlur(m, (0, 0), sig) + 1e-6)

    dog = nb(s1) - nb(s2)
    inner = cv2.erode(m.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    v = dog[inner]
    return float(np.percentile(v, 99)) if v.size > 20 else np.nan


def local_contrast(th_c, th_m, x, y, inner, outer, hand_med):
    h, w = th_c.shape
    r0, r1, c0, c1 = max(0, y - inner), min(h, y + inner + 1), max(0, x - inner), min(w, x + inner + 1)
    sel = th_m[r0:r1, c0:c1] > 0
    if sel.sum() < 5:
        return np.nan, np.nan, False
    loc = float(np.median(th_c[r0:r1, c0:c1][sel]))
    R0, R1, C0, C1 = max(0, y - outer), min(h, y + outer + 1), max(0, x - outer), min(w, x + outer + 1)
    ring = th_m[R0:R1, C0:C1] > 0
    ring[max(0, r0 - R0):max(0, r1 - R0), max(0, c0 - C0):max(0, c1 - C0)] = False
    sur = float(np.median(th_c[R0:R1, C0:C1][ring])) if ring.sum() >= 10 else hand_med
    return loc, loc - sur, True


def frame_stats(th_c, th_m, th_dil, pts, scale) -> dict | None:
    sel = th_m > 0
    px = th_c[sel]
    if px.size < 200:
        return None
    med = float(np.median(px))
    gy, gx = np.gradient(th_c)
    grad_norm = float(np.hypot(gx, gy)[sel].mean()) * max(1.0, scale)
    st = {
        "mean": float(px.mean()),
        "p90": float(np.percentile(px, 90)),
        "sd": float(px.std()),
        "grad_norm": grad_norm,
        "p99_minus_med": float(np.percentile(px, 99) - med),
        "focal_peak": focal_peak_c(th_c, th_m, scale),
        "j_valid": 0.0,
        "j_contrast": np.nan,
        "j_p90_contrast": np.nan,
        "j_peak": np.nan,
        "j_palm_tip": np.nan,
    }
    for n in FINGER_CHAINS:
        st[f"chain_{n}"] = np.nan
    if pts is not None:
        inner, outer = int(max(3, 0.025 * scale)), int(max(9, 0.08 * scale))
        temps, contr, valid = np.full(21, np.nan), np.full(21, np.nan), np.zeros(21, bool)
        for j in range(21):
            x, y = int(round(pts[j, 0])), int(round(pts[j, 1]))
            if not (0 <= x < 640 and 0 <= y < 480) or th_dil[y, x] == 0:
                continue
            temps[j], contr[j], valid[j] = local_contrast(th_c, th_m, x, y, inner, outer, med)
        st["j_valid"] = float(valid.mean())
        if valid.any():
            c_fin = contr[valid][np.isfinite(contr[valid])]
            if c_fin.size:
                st["j_contrast"] = float(np.mean(c_fin))
                st["j_p90_contrast"] = float(np.percentile(c_fin, 90))
                st["j_peak"] = float(np.max(c_fin))
            palm = [temps[i] for i in PALM_IDXS if np.isfinite(temps[i])]
            tips = [temps[i] for i in (4, 8, 12, 16, 20) if np.isfinite(temps[i])]
            if palm and tips:
                st["j_palm_tip"] = float(np.mean(palm) - np.mean(tips))
            for n, ch in FINGER_CHAINS.items():
                if np.isfinite(temps[ch[0]]) and np.isfinite(temps[ch[-1]]):
                    st[f"chain_{n}"] = float(temps[ch[0]] - temps[ch[-1]])
    return st


def aggregate(frames: list[dict]) -> dict:
    if len(frames) < 3:
        return {}

    def col(key):
        v = np.array([f[key] for f in frames], float)
        return v[np.isfinite(v)]

    m = np.array([f["mean"] for f in frames])
    out = {
        "f_mask_mean_c": float(m.mean()),
        "f_mask_p90_c": float(col("p90").mean()),
        "f_mask_sd_c": float(col("sd").mean()),
        "f_mask_grad_norm_c": float(col("grad_norm").mean()),
        "f_mask_p99_minus_med_c": float(col("p99_minus_med").mean()),
        "f_focal_peak_c": float(col("focal_peak").mean()) if col("focal_peak").size else np.nan,
        "f_drift_slope_c": float(np.polyfit(np.arange(len(m)), m, 1)[0]),
        "f_step_noise_c": float(np.median(np.abs(np.diff(m)))),
    }
    joint_map = [
        ("f_local_joint_contrast_c", "j_contrast"),
        ("f_joint_p90_contrast_c", "j_p90_contrast"),
        ("f_peak_joint_excess_c", "j_peak"),
        ("f_palm_minus_tip_c", "j_palm_tip"),
    ] + [(f"f_{n}_mcp_minus_tip_c", f"chain_{n}") for n in FINGER_CHAINS]
    for ok_, key in joint_map:
        v = col(key)
        out[ok_] = float(v.mean()) if v.size >= 3 else np.nan
    return out


def inject_perturbation(th_c, th_m, delta, y_pct, scale, mode="focal_blob"):
    ys, xs = np.nonzero(th_m)
    if xs.size == 0 or abs(delta) <= 1e-9:
        return th_c
    if mode == "whole_hand":
        return (th_c + delta * (th_m > 0)).astype(np.float32)
    ty = np.percentile(ys, y_pct)
    band = np.abs(ys - ty) <= max(3.0, 0.03 * scale)
    bx, by = (xs[band], ys[band]) if band.any() else (xs, ys)
    j = int(np.argmin(np.abs(bx - np.median(bx))))
    cx, cy = float(bx[j]), float(by[j])
    sigma = max(6.0, 0.06 * scale)
    yy, xx = np.ogrid[:480, :640]
    blob = delta * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2.0 * sigma ** 2))
    return (th_c + blob * (th_m > 0)).astype(np.float32)


def process_recording(npy_path, rgb_path, rec_id, ctx) -> dict:
    th_raw = np.load(npy_path, mmap_mode="r")
    n_th = int(th_raw.shape[0])
    idxs = np.arange(0, n_th, max(1, ctx["stride"]))
    if len(idxs) > ctx["max_frames"]:
        idxs = idxs[np.linspace(0, len(idxs) - 1, ctx["max_frames"]).astype(int)]
    use_j = ctx["det"] is not None
    cap = cv2.VideoCapture(str(rgb_path)) if (use_j and rgb_path is not None) else None
    n_rgb = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if (cap is not None and cap.isOpened()) else 0

    variants = [("focal", d) for d in FOCAL_DELTAS] + [("whole_hand", d) for d in REGIONAL_DELTAS]
    per_var: dict[tuple[str, float], list[dict]] = {("clean", 0.0): []}
    for kind, d in variants:
        per_var[(kind, d)] = []

    y_pct = (20, 35, 50)[zlib.crc32(rec_id.encode()) % 3]
    n_seen = n_ok = attempts = hits = in_strict = in_tot = n_unet = 0
    areas = []

    for ti in idxs:
        th_c = orient_thermal(to_celsius(th_raw[int(ti)]), ctx["orient"])
        th_m, src = get_mask(th_c, ctx["mask_source"], ctx["seg"], ctx["device"])
        n_seen += 1
        areas.append(float(th_m.mean()))
        if not mask_ok(th_m):
            continue
        n_ok += 1
        n_unet += int(src == "unet")
        pts = None
        if cap is not None and cap.isOpened() and n_rgb > 0:
            ri = int(round(ti / max(1, n_th - 1) * (n_rgb - 1)))
            rgb = read_rgb(cap, ri)
            if rgb is not None:
                attempts += 1
                raw = detect_pts(ctx["det"], canon_rgb(rgb, ctx["rgb_k"]))
                if raw is not None:
                    hits += 1
                    pts = cv2.transform(raw.reshape(-1, 1, 2), ctx["M"]).reshape(-1, 2)
                    xi, yi = np.round(pts[:, 0]).astype(int), np.round(pts[:, 1]).astype(int)
                    inb = (xi >= 0) & (xi < 640) & (yi >= 0) & (yi < 480)
                    in_strict += int((th_m[yi[inb], xi[inb]] > 0).sum())
                    in_tot += 21
        scale = float(np.sqrt(th_m.sum()))
        th_dil = cv2.dilate(th_m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)))

        st_clean = frame_stats(th_c, th_m, th_dil, pts, scale)
        if st_clean is not None:
            per_var[("clean", 0.0)].append(st_clean)
        for kind, d in variants:
            th_v = inject_perturbation(th_c, th_m, d, y_pct, scale, mode="whole_hand" if kind == "whole_hand" else "focal_blob")
            st_v = frame_stats(th_v, th_m, th_dil, pts, scale)
            if st_v is not None:
                per_var[(kind, d)].append(st_v)

    if cap is not None:
        cap.release()

    clean = per_var[("clean", 0.0)]
    quality = {
        "n_used": len(clean),
        "q_mask_ok_frac": n_ok / max(1, n_seen),
        "q_area_cv": float(np.std(areas) / max(1e-6, np.mean(areas))) if areas else np.nan,
        "q_mp_hit_rate": hits / attempts if attempts else np.nan,
        "q_valid_joint_frac": float(np.mean([f["j_valid"] for f in clean])) if clean else 0.0,
        "q_joint_in_mask_strict_pct": 100.0 * in_strict / in_tot if in_tot else np.nan,
        "q_unet_frac": n_unet / max(1, n_ok),
    }
    agg_variants = {"clean_+0.0C": aggregate(clean)}
    for kind, d in variants:
        agg_variants[f"{kind}_+{d}C"] = aggregate(per_var[(kind, d)])
    return {"quality": quality, "variants": agg_variants}


# ----------------------------------------------------------------------------- scorer
class SubspaceResidualScorer:
    """In-subspace MCD Mahalanobis (optional) + orthogonal residual (SPE). Robust-scaled z inputs."""

    def __init__(self, seed: int = 42, spe_weight: float = 0.5, use_pca: bool = True) -> None:
        self.seed = seed
        self.spe_weight = float(spe_weight)
        self.use_pca = bool(use_pca)

    def _prep(self, X):
        X = np.asarray(X, float)
        return np.where(np.isfinite(X), X, self.impute)

    def _z(self, Xi):
        return np.clip((Xi - self.center) / self.scale, -6.0, 6.0)

    def _spe(self, Z, P):
        if not self.use_pca:
            return np.zeros(len(Z))
        return np.sqrt(np.mean((Z - self.pca.inverse_transform(P)) ** 2, axis=1))

    def fit(self, X):
        X = np.asarray(X, float)
        med = np.nanmedian(X, 0)
        self.impute = np.where(np.isfinite(med), med, 0.0)
        Xi = self._prep(X)
        self.center = np.median(Xi, 0)
        mad = 1.4826 * np.median(np.abs(Xi - self.center), 0)
        self.scale = np.maximum(np.maximum(mad, 0.5 * Xi.std(0)), 1e-3)
        Z = self._z(Xi)
        if self.use_pca:
            cum = np.cumsum(PCA(random_state=self.seed).fit(Z).explained_variance_ratio_)
            hi = max(2, min(6, Z.shape[1], Z.shape[0] - 2))
            nc = int(np.clip(np.searchsorted(cum, 0.90) + 1, 2, hi))
            self.pca = PCA(n_components=nc, random_state=self.seed).fit(Z)
            P = self.pca.transform(Z)
        else:
            P = Z
        try:
            self.cov = MinCovDet(support_fraction=0.8, random_state=self.seed).fit(P)
        except Exception:
            self.cov = EmpiricalCovariance().fit(P)
        d_sub = np.sqrt(np.maximum(0.0, self.cov.mahalanobis(P)))
        d_res = self._spe(Z, P)
        self.sub_med = float(np.median(d_sub))
        self.sub_iqr = max(1e-3, float(np.percentile(d_sub, 75) - np.percentile(d_sub, 25)))
        self.res_med = float(np.median(d_res))
        self.res_iqr = max(1e-3, float(np.percentile(d_res, 75) - np.percentile(d_res, 25)))
        return self

    def components(self, X):
        Z = self._z(self._prep(X))
        P = self.pca.transform(Z) if self.use_pca else Z
        d_sub = np.sqrt(np.maximum(0.0, self.cov.mahalanobis(P)))
        d_res = self._spe(Z, P)
        z_sub = (d_sub - self.sub_med) / self.sub_iqr
        z_res = (d_res - self.res_med) / self.res_iqr
        w = self.spe_weight if self.use_pca else 0.0
        d_comb = np.sqrt(np.maximum(0.0, z_sub) ** 2 + w * np.maximum(0.0, z_res) ** 2)
        return d_comb, d_sub, d_res

    def score(self, X):
        return self.components(X)[0]

    def score_z_rms(self, X):
        return np.sqrt(np.mean(self._z(self._prep(X)) ** 2, axis=1))

    def to_bundle(self) -> dict:
        b = {
            "impute": self.impute, "center": self.center, "scale": self.scale, "clip_z": 6.0,
            "mcd_location": self.cov.location_, "mcd_precision": self.cov.precision_,
            "sub_med": self.sub_med, "sub_iqr": self.sub_iqr,
            "res_med": self.res_med, "res_iqr": self.res_iqr,
            "spe_weight": self.spe_weight, "use_pca": self.use_pca,
        }
        if self.use_pca:
            b["pca_mean"], b["pca_components"] = self.pca.mean_, self.pca.components_
        return b


def score_from_bundle(bundle: dict, feats: dict) -> float:
    """Pure-numpy reference scorer for export/mobile parity (mirrors SubspaceResidualScorer.components)."""
    x = np.array([np.nan if feats.get(c) is None else float(feats[c]) for c in bundle["feature_columns"]])
    x = np.where(np.isfinite(x), x, np.array(bundle["impute"], float))
    z = np.clip((x - np.array(bundle["center"])) / np.array(bundle["scale"]), -bundle["clip_z"], bundle["clip_z"])
    use_pca = bool(bundle.get("use_pca", True))
    loc = np.array(bundle["mcd_location"], float)
    prec = np.array(bundle["mcd_precision"], float)
    if use_pca:
        pm = np.array(bundle["pca_mean"], float)
        pc = np.array(bundle["pca_components"], float)
        p = (z - pm) @ pc.T
        z_hat = pm + p @ pc
        d_res_raw = float(np.sqrt(np.mean((z - z_hat) ** 2)))
    else:
        p = z
        d_res_raw = 0.0
    dd = p - loc
    d_sub_raw = float(np.sqrt(max(0.0, dd @ prec @ dd)))
    z_sub = max(0.0, (d_sub_raw - float(bundle["sub_med"])) / float(bundle["sub_iqr"]))
    z_res = max(0.0, (d_res_raw - float(bundle["res_med"])) / float(bundle["res_iqr"]))
    w = float(bundle.get("spe_weight", 0.5)) if use_pca else 0.0
    return float(np.sqrt(z_sub ** 2 + w * (z_res ** 2)))


# ----------------------------------------------------------------------------- tabular helpers
def add_lr(df: pd.DataFrame, lr_base: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in lr_base:
        out[f"{c}_lr_abs"] = np.nan
    for _, g in out.groupby(["subject", "task"]):
        L, R = g[g.hand == "left"], g[g.hand == "right"]
        if len(L) and len(R):
            for c in lr_base:
                a, b = float(L[c].iloc[0]), float(R[c].iloc[0])
                out.loc[g.index, f"{c}_lr_abs"] = abs(a - b) if np.isfinite(a) and np.isfinite(b) else np.nan
    return out


def pick_targets(df: pd.DataFrame) -> np.ndarray:
    tg = np.zeros(len(df), bool)
    for (s, t), g in df.groupby(["subject", "task"]):
        want = "left" if zlib.crc32(f"{s}_{t}".encode()) % 2 == 0 else "right"
        hands = set(g.hand)
        for i, h in zip(g.index, g.hand):
            tg[i] = (h == want) or (want not in hands)
    return tg


def boot_auc(neg, neg_s, pos, pos_s, n_boot, rng):
    subs = np.unique(np.concatenate([neg_s, pos_s]))
    nb = {s: neg[neg_s == s] for s in subs}
    pb = {s: pos[pos_s == s] for s in subs}
    vals = []
    for _ in range(n_boot):
        pick = rng.choice(subs, len(subs), replace=True)
        n = np.concatenate([nb[s] for s in pick])
        p = np.concatenate([pb[s] for s in pick])
        if n.size and p.size:
            vals.append(roc_auc_score(np.r_[np.zeros(n.size), np.ones(p.size)], np.r_[n, p]))
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))] if vals else [np.nan, np.nan]


def detection_metrics(d_clean, d_inj, df, tgt, n_boot, rng) -> dict:
    ok = df.gate_ok.to_numpy() & np.isfinite(d_clean)
    subj = df.subject.to_numpy()
    pos_m = tgt & ok & np.isfinite(d_inj)
    neg, neg_s, pos, pos_s = d_clean[ok], subj[ok], d_inj[pos_m], subj[pos_m]
    if neg.size == 0 or pos.size == 0:
        return {}
    cut = float(np.percentile(neg, 95))
    return {
        "pooled_auroc": float(roc_auc_score(np.r_[np.zeros(neg.size), np.ones(pos.size)], np.r_[neg, pos])),
        "pooled_auroc_ci95_subject_bootstrap": boot_auc(neg, neg_s, pos, pos_s, n_boot, rng),
        "paired_auroc_same_recording": float(roc_auc_score(
            np.r_[np.zeros(pos.size), np.ones(pos.size)], np.r_[d_clean[pos_m], d_inj[pos_m]])),
        "paired_win_rate": float(np.mean(d_inj[pos_m] > d_clean[pos_m])),
        "tpr_at_oof_p95_cut": float(np.mean(pos > cut)),
        "n_pos": int(pos.size),
        "n_neg": int(neg.size),
    }


def label_shuffle_control(d_clean, d_inj, df, tgt, seed, n_rep=200) -> dict:
    """Permute clean-vs-injected labels on the same scores. Null AUROC must be ~0.5 by construction."""
    ok = df.gate_ok.to_numpy() & np.isfinite(d_clean)
    pos_m = tgt & ok & np.isfinite(d_inj)
    pool = np.r_[d_clean[ok], d_inj[pos_m]]
    n_neg, n_pos = int(ok.sum()), int(pos_m.sum())
    if n_neg == 0 or n_pos == 0:
        return {}
    rng = np.random.default_rng(seed)
    base = np.r_[np.zeros(n_neg), np.ones(n_pos)]
    aucs = [roc_auc_score(rng.permutation(base), pool) for _ in range(n_rep)]
    return {"mean_auroc": float(np.mean(aucs)),
            "ci95": [float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))]}


def evaluate_loso(df_eval, dfc, inj_dfs, cols, tgt, seed, n_boot, use_pca=True):
    n = len(df_eval)
    d_comb, d_sub, d_res, d_z = np.full(n, np.nan), np.full(n, np.nan), np.full(n, np.nan), np.full(n, np.nan)
    i_comb = {k: np.full(n, np.nan) for k in inj_dfs}
    i_z = {k: np.full(n, np.nan) for k in inj_dfs}
    fold_cut = {}

    for s in sorted(df_eval.subject.unique()):
        tr = ((df_eval.subject != s) & df_eval.gate_ok).to_numpy()
        te = (df_eval.subject == s).to_numpy()
        if tr.sum() < MIN_FIT_RECORDINGS:
            continue
        sc = SubspaceResidualScorer(seed, use_pca=use_pca).fit(dfc.loc[tr, cols].to_numpy(float))
        fold_cut[s] = float(np.percentile(sc.score(dfc.loc[tr, cols].to_numpy(float)), 95))
        c_te, s_te, r_te = sc.components(dfc.loc[te, cols].to_numpy(float))
        d_comb[te], d_sub[te], d_res[te] = c_te, s_te, r_te
        d_z[te] = sc.score_z_rms(dfc.loc[te, cols].to_numpy(float))
        for k in inj_dfs:
            X = inj_dfs[k].loc[te, cols].to_numpy(float)
            i_comb[k][te] = sc.score(X)
            i_z[k][te] = sc.score_z_rms(X)

    rng = np.random.default_rng(seed)
    ok = df_eval.gate_ok.to_numpy() & np.isfinite(d_comb)
    cut_final = float(np.percentile(d_comb[ok], 95)) if ok.any() else np.nan
    subj_arr = df_eval.subject.to_numpy()
    fl = {s: ((d_comb[ok & (subj_arr == s)] > fold_cut.get(s, np.inf)).sum(), (ok & (subj_arr == s)).sum())
          for s in fold_cut}
    subs = list(fl)
    boots = []
    for _ in range(n_boot):
        pick = rng.choice(subs, len(subs), replace=True)
        k, m = sum(fl[s][0] for s in pick), sum(fl[s][1] for s in pick)
        boots.append(k / max(1, m))
    flag_rate = sum(v[0] for v in fl.values()) / max(1, sum(v[1] for v in fl.values()))

    fault = {k: {
        "subspace_spe" if use_pca else "mcd_no_pca": detection_metrics(d_comb, i_comb[k], df_eval, tgt, n_boot, rng),
        "z_rms_baseline": detection_metrics(d_z, i_z[k], df_eval, tgt, n_boot, rng),
    } for k in inj_dfs}

    return {
        "d_comb": d_comb, "d_sub": d_sub, "d_res": d_res, "d_z": d_z,
        "i_comb": i_comb, "cut_final": cut_final,
        "clean_flag_rate_fold_train_cut": float(flag_rate),
        "clean_flag_rate_ci95": [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))] if boots else [np.nan, np.nan],
        "fault_injection": fault,
    }


# ----------------------------------------------------------------------------- main
def main(args) -> dict:
    t0 = time.perf_counter()
    data_root, work = Path(args.data_root), Path(args.work_root)
    work.mkdir(parents=True, exist_ok=True)
    npy_files, mp4_files = sorted(data_root.rglob("*.npy")), sorted(data_root.rglob("*.mp4"))
    if not npy_files:
        raise FileNotFoundError(f"No .npy under {data_root}")
    meta = [parse_recording_meta(p, i) for i, p in enumerate(npy_files)]
    print(f"[DATA] {len(npy_files)} recordings, {len({m[0] for m in meta})} subjects, {len(mp4_files)} videos", flush=True)
    if args.limit:
        npy_files, meta = npy_files[: args.limit], meta[: args.limit]

    det = None if args.no_joints else make_detector(Path(args.task_path))
    device = torch.device("cuda" if HAS_TORCH and torch.cuda.is_available() else "cpu") if HAS_TORCH else None
    seg = load_seg_model(Path(args.seg_ckpt), device) if args.mask_source != "otsu" else None
    try:
        rgb_k, rot_rates = 1, {}
        reg = (
            fit_registration(npy_files, mp4_files, meta, det, rgb_k, args.seed)
            if det is not None
            else {"M": make_M(1.97, 0, -211.0, -296.0), "orient": "rot_ccw", "joints_ok": False}
        )
        (work / "registration.json").write_text(json.dumps(_jsonable({
            "M": reg["M"], "orient": reg["orient"], "joints_ok": reg["joints_ok"],
            "heldout": reg.get("heldout")}), indent=2), encoding="utf-8")
        joints_on = bool(det is not None and (reg["joints_ok"] or args.force_joints))
        ctx = {
            "M": reg["M"], "orient": reg["orient"], "rgb_k": rgb_k,
            "det": det if joints_on else None, "mask_source": args.mask_source, "seg": seg,
            "device": device, "stride": args.stride, "max_frames": args.max_frames,
        }
        cache = work / "features_cache.pkl"
        cfg = {
            "schema": "v6_hysteresis_mask", "stride": args.stride, "max_frames": args.max_frames,
            "mask": args.mask_source, "joints": joints_on, "orient": reg["orient"], "rgb_k": rgb_k,
            "n": len(npy_files), "M": np.round(reg["M"], 3).tolist(),
        }
        rows = None
        if args.reuse_features and cache.exists():
            blob = pickle.loads(cache.read_bytes())
            if blob.get("cfg") == cfg:
                rows = blob["rows"]
                print("[CACHE] reusing extracted features", flush=True)
        if rows is None:
            rows = []
            for i, (np_p, (subj, hand, task)) in enumerate(zip(npy_files, meta)):
                rid = f"{subj}_{hand}_{task}_{i:02d}"
                rgb_p = find_rgb_for_npy(subj, hand, task, mp4_files)
                rows.append({"rec_id": rid, "subject": subj, "hand": hand, "task": task,
                             **process_recording(np_p, rgb_p, rid, ctx)})
                if (i + 1) % 10 == 0 or i + 1 == len(npy_files):
                    print(f"[FEAT {i + 1}/{len(npy_files)}] {time.perf_counter() - t0:.0f}s", flush=True)
            cache.write_bytes(pickle.dumps({"cfg": cfg, "rows": rows}))
    finally:
        if det is not None:
            det.close()

    df = pd.DataFrame([
        {"rec_id": r["rec_id"], "subject": r["subject"], "hand": r["hand"], "task": r["task"],
         **r["quality"], **r["variants"]["clean_+0.0C"]}
        for r in rows
    ]).reset_index(drop=True)

    joint_gate = (df.q_valid_joint_frac >= GATE_MIN_VALID_JOINT_FRAC) if joints_on else True
    df["gate_ok"] = (
        (df.n_used >= 3) & (df.q_mask_ok_frac >= GATE_MIN_MASK_OK_FRAC)
        & (df.q_area_cv <= GATE_MAX_AREA_CV) & joint_gate
    )
    print(f"[GATE] pre-registered gate_pass_rate={df.gate_ok.mean():.1%} ({int(df.gate_ok.sum())}/{len(df)})", flush=True)
    if df.gate_ok.sum() < MIN_FIT_RECORDINGS:
        raise RuntimeError(f"Only {int(df.gate_ok.sum())} gate-passing recordings; refusing to fit a cohort model.")

    feat_cols = []
    for c in [c for c in df.columns if c.startswith("f_")]:
        v = df[c].to_numpy(float)
        if np.isfinite(v).mean() >= 0.7 and np.nanstd(v) > 1e-6:
            feat_cols.append(c)
        else:
            print(f"[FEAT] dropped {c} (missing/constant)", flush=True)

    lr_base = [c for c in LR_BASE if c in feat_cols]
    cols = feat_cols + [f"{c}_lr_abs" for c in lr_base]
    dfc = add_lr(df, lr_base)
    tgt = pick_targets(df)

    variant_keys = [f"focal_+{d}C" for d in FOCAL_DELTAS] + [f"whole_hand_+{d}C" for d in REGIONAL_DELTAS]
    inj_dfs: dict[str, pd.DataFrame] = {}
    for vk in variant_keys:
        raw_v = pd.DataFrame([r["variants"][vk] for r in rows]).reindex(columns=feat_cols)
        dfi = df.copy()
        dfi.loc[tgt, feat_cols] = raw_v.loc[tgt, feat_cols].to_numpy()
        inj_dfs[vk] = add_lr(dfi, lr_base)

    main_eval = evaluate_loso(df, dfc, inj_dfs, cols, tgt, args.seed, args.boot, use_pca=True)
    ablation_eval = evaluate_loso(df, dfc, inj_dfs, cols, tgt, args.seed, args.boot, use_pca=False)
    d_comb, d_sub, d_res, d_z = main_eval["d_comb"], main_eval["d_sub"], main_eval["d_res"], main_eval["d_z"]
    cut_final = main_eval["cut_final"]

    # Negative control: shuffle subject IDs (retained for continuity; NOT a chance-level proof)
    rng_shuf = np.random.default_rng(args.seed + 1000)
    df_shuf = df.copy()
    df_shuf["subject"] = rng_shuf.permutation(df["subject"].to_numpy())
    shuf_eval = evaluate_loso(df_shuf, dfc, inj_dfs, cols, tgt, args.seed, max(100, args.boot // 2), use_pca=True)

    # Valid chance-level control: permute clean-vs-injected labels on identical scores
    label_ctrl = {k: label_shuffle_control(d_comb, main_eval["i_comb"][k], df, tgt, args.seed) for k in inj_dfs}

    f1_report: dict[str, Any] = {"real_labels_provided": False,
                                 "note": "No ground-truth labels provided; cohort deviation is unsupervised."}
    if args.labels_csv and Path(args.labels_csv).exists():
        ldf = pd.read_csv(args.labels_csv)
        if {"rec_id", "label"}.issubset(ldf.columns):
            m_df = df[["rec_id", "subject"]].assign(d=d_comb).merge(ldf[["rec_id", "label"]], on="rec_id", how="inner")
            m_df = m_df[np.isfinite(m_df.d)]
            if len(m_df) >= 6 and m_df["label"].nunique() >= 2:
                y_all = m_df["label"].astype(int).to_numpy()
                s_all = m_df["subject"].to_numpy()
                d_all = m_df["d"].to_numpy()
                oof_pred = np.zeros(len(m_df), dtype=int)
                for s in np.unique(s_all):
                    tr_m, te_m = s_all != s, s_all == s
                    best_f1, best_th = -1.0, float(np.median(d_all[tr_m]))
                    for th in np.quantile(d_all[tr_m], np.linspace(0.1, 0.9, 33)):
                        f = f1_score(y_all[tr_m], (d_all[tr_m] >= th).astype(int), zero_division=0)
                        if f > best_f1:
                            best_f1, best_th = float(f), float(th)
                    oof_pred[te_m] = (d_all[te_m] >= best_th).astype(int)
                f1_report = {
                    "real_labels_provided": True,
                    "oof_f1": round(float(f1_score(y_all, oof_pred, zero_division=0)), 4),
                    "oof_precision": round(float(precision_score(y_all, oof_pred, zero_division=0)), 4),
                    "oof_recall": round(float(recall_score(y_all, oof_pred, zero_division=0)), 4),
                }

    ok = df.gate_ok.to_numpy() & np.isfinite(d_comb)
    results: dict[str, Any] = {
        "pipeline_version": "final_v6_hysteresis",
        "model_type": "subspace_mcd_plus_spe_residual",
        "n_recordings": len(df),
        "n_subjects": int(df.subject.nunique()),
        "quality_gate_rule": {
            "min_n_used": 3, "min_mask_ok_frac": GATE_MIN_MASK_OK_FRAC, "max_area_cv": GATE_MAX_AREA_CV,
            "min_valid_joint_frac_when_joints_on": GATE_MIN_VALID_JOINT_FRAC if joints_on else None,
        },
        "gate_pass_rate": float(df.gate_ok.mean()),
        "mask_source": args.mask_source,
        "mediapipe_active": bool(ctx["det"] is not None),
        "rgb_rotation_k": rgb_k,
        "rgb_rotation_hit_rates": rot_rates,
        "thermal_orient": reg["orient"],
        "registration": {k: v for k, v in reg.items() if k != "M"},
        "joint_features_enabled": joints_on,
        "mp_hit_rate_mean": float(df.q_mp_hit_rate.mean()),
        "joint_in_mask_strict_pct_mean": float(df.q_joint_in_mask_strict_pct.mean()),
        "features_used": cols,
        "clean_flag_rate_fold_train_cut": main_eval["clean_flag_rate_fold_train_cut"],
        "clean_flag_rate_ci95": main_eval["clean_flag_rate_ci95"],
        "fault_injection": main_eval["fault_injection"],
        "ablation_pca_contribution": {
            "with_pca_spe": main_eval["fault_injection"],
            "without_pca": ablation_eval["fault_injection"],
        },
        "negative_control_label_shuffle": label_ctrl,
        "negative_control_subject_shuffle": {
            "clean_flag_rate_fold_train_cut": shuf_eval["clean_flag_rate_fold_train_cut"],
            "fault_injection": shuf_eval["fault_injection"],
        },
        "supervised_f1": f1_report,
        "d_vs_quality_spearman": {
            q: float(spearmanr(d_comb[ok], df.loc[ok, q], nan_policy="omit").correlation)
            for q in ("q_mask_ok_frac", "q_area_cv", "q_valid_joint_frac")
        },
        "disclaimer": DISCLAIMER,
    }

    piv = pd.DataFrame({"s": df.subject, "h": df.hand, "t": df.task, "d": d_comb}).pivot_table(
        index=["s", "h"], columns="t", values="d")
    if {"static", "open_close"}.issubset(piv.columns) and piv.dropna().shape[0] >= 5:
        results["repeatability_spearman_static_vs_open_close"] = float(
            spearmanr(piv["static"], piv["open_close"], nan_policy="omit").correlation)

    fit_mask = df.gate_ok.to_numpy()
    final = SubspaceResidualScorer(args.seed, use_pca=True).fit(dfc.loc[fit_mask, cols].to_numpy(float))
    bundle = {
        "schema_version": "6.0.0",
        "model_type": "subspace_mcd_plus_spe_residual",
        "feature_columns": cols,
        **final.to_bundle(),
        "d_cut_oof_p95": cut_final,
        "thermal_orient": reg["orient"],
        "rgb_rotation_k": rgb_k,
        "M_global": reg["M"],
        "canonical_hw": [480, 640],
        "joint_features_enabled": joints_on,
        "mask_source": args.mask_source,
        "disclaimer": DISCLAIMER,
    }
    chk = dfc.loc[fit_mask].head(5)
    par = [
        abs(score_from_bundle(_jsonable(bundle), r.to_dict()) - float(final.score(r[cols].to_numpy(float)[None])[0]))
        for _, r in chk.iterrows()
    ]
    results["bundle_parity_max_abs_diff"] = float(max(par)) if par else None

    out = df[["rec_id", "subject", "hand", "task", "gate_ok", "q_mask_ok_frac", "q_area_cv",
              "q_mp_hit_rate", "q_valid_joint_frac"]].copy()
    out["d_loso_comb"] = np.round(d_comb, 4)
    out["d_loso_subspace"] = np.round(d_sub, 4)
    out["d_loso_residual_spe"] = np.round(d_res, 4)
    out["d_z_rms_baseline"] = np.round(d_z, 4)
    out["status"] = np.where(~df.gate_ok, "RECAPTURE_RECOMMENDED",
                             np.where(d_comb > cut_final, "OUTLIER_VS_COHORT", "WITHIN_COHORT_RANGE"))
    out["disclaimer"] = DISCLAIMER
    out.to_csv(work / "loso_predictions.csv", index=False)
    (work / "final_v6_bundle.json").write_text(json.dumps(_jsonable(bundle), indent=2))
    results["elapsed_sec"] = round(time.perf_counter() - t0, 1)
    (work / "results.json").write_text(json.dumps(_jsonable(results), indent=2))
    print(json.dumps(_jsonable({k: v for k, v in results.items() if k not in ("features_used",)}), indent=2), flush=True)
    return results


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", default=DATA_ROOT)
    ap.add_argument("--work-root", default=WORK_ROOT)
    ap.add_argument("--seg-ckpt", default=SEG_CKPT)
    ap.add_argument("--task-path", default=TASK_PATH)
    ap.add_argument("--labels-csv", default="")
    ap.add_argument("--mask-source", choices=["otsu", "auto"], default="otsu")
    ap.add_argument("--stride", type=int, default=2)
    ap.add_argument("--max-frames", type=int, default=16)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--boot", type=int, default=500)
    ap.add_argument("--limit", type=int, default=0, help="process first N recordings (pipeline smoke test only)")
    ap.add_argument("--reuse-features", action="store_true")
    ap.add_argument("--no-joints", action="store_true")
    ap.add_argument("--force-joints", action="store_true")
    main(ap.parse_known_args()[0])

[DATA] 68 recordings, 17 subjects, 136 videos


INFO: [environment.cc:36] Creating LiteRT environment with options
INFO: [accelerator_registry.cc:54] RegisterAccelerator: ptr=0x785ef0003ce0, name=CpuAccelerator
INFO: [cpu_registry.cc:75] XNNPACK CPU accelerator registered.
INFO: [compiled_model.cc:1022] Flatbuffer model initialized directly from incoming litert model.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: [environment.cc:36] Creating LiteRT environment with options
INFO: [accelerator_registry.cc:54] RegisterAccelerator: ptr=0x785efc034620, name=CpuAccelerator
INFO: [cpu_registry.cc:75] XNNPACK CPU accelerator registered.
INFO: [compiled_model.cc:1022] Flatbuffer model initialized directly from incoming litert model.
W0000 00:00:1795285480.630749     167 inference_runner_litert.cc:1433] Input tensor memory is not aligned according to LiteRt's memory alignment requirements. Reallocating memory and copying data.
W0000 00:00:1795285480.676077     168 landmark_projection_calculator.cc:81] Using NORM_RECT without I

[REG] orient=rot_ccw heldout IoU=0.584 joints-in-mask(tol)=86.3% -> joints ENABLED
[FEAT 10/68] 224s
[FEAT 20/68] 280s
[FEAT 30/68] 337s
[FEAT 40/68] 392s
[FEAT 50/68] 448s
[FEAT 60/68] 504s
[FEAT 68/68] 549s
[GATE] pre-registered gate_pass_rate=97.1% (66/68)
[FEAT] dropped f_thumb_mcp_minus_tip_c (missing/constant)


INFO: [accelerator_registry.cc:43] DestroyAccelerator: ptr=0x785ef0003ce0, name=CpuAccelerator
INFO: [accelerator_registry.cc:43] DestroyAccelerator: ptr=0x785efc034620, name=CpuAccelerator


{
  "pipeline_version": "final_v6_hysteresis",
  "model_type": "subspace_mcd_plus_spe_residual",
  "n_recordings": 68,
  "n_subjects": 17,
  "quality_gate_rule": {
    "min_n_used": 3,
    "min_mask_ok_frac": 0.85,
    "max_area_cv": 0.35,
    "min_valid_joint_frac_when_joints_on": 0.5
  },
  "gate_pass_rate": 0.9705882352941176,
  "mask_source": "otsu",
  "mediapipe_active": true,
  "rgb_rotation_k": 1,
  "rgb_rotation_hit_rates": {},
  "thermal_orient": "rot_ccw",
  "registration": {
    "orient": "rot_ccw",
    "joints_ok": true,
    "mp_hit_rate_calibration": 0.9656862745098039,
    "n_pairs": 197,
    "heldout": {
      "n_pairs": 78,
      "median_iou": 0.583820450889311,
      "joint_in_mask_tol_pct": 86.26373626373626
    },
    "per_orient": {
      "rot_ccw": {
        "fit_score": 0.6911620634005127,
        "heldout_n_pairs": 78,
        "heldout_median_iou": 0.583820450889311,
        "heldout_joint_in_mask_tol_pct": 86.26373626373626
      },
      "rot_cw": {
        "fi

In [3]:
import json
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.covariance import EmpiricalCovariance, MinCovDet
from sklearn.decomposition import PCA

DISCLAIMER = "relative to the 17-subject cohort, not a diagnosis"
LR_BASE = [
    "f_mask_mean_c",
    "f_focal_peak_c",
    "f_local_joint_contrast_c",
    "f_joint_p90_contrast_c",
    "f_palm_minus_tip_c",
]


class CohortScorer:
    def __init__(self, seed: int = 42, spe_weight: float = 0.5) -> None:
        self.seed = seed
        self.spe_weight = spe_weight

    def fit(self, X: np.ndarray):
        X = np.asarray(X, dtype=np.float64)
        med = np.nanmedian(X, axis=0)
        self.impute = np.where(np.isfinite(med), med, 0.0)
        Xi = np.where(np.isfinite(X), X, self.impute)
        self.center = np.median(Xi, axis=0)
        mad = 1.4826 * np.median(np.abs(Xi - self.center), axis=0)
        self.scale = np.maximum(np.maximum(mad, 0.5 * np.std(Xi, axis=0)), 1e-3)
        Z = np.clip((Xi - self.center) / self.scale, -6.0, 6.0)

        cum = np.cumsum(PCA(random_state=self.seed).fit(Z).explained_variance_ratio_)
        hi = max(2, min(6, Z.shape[1], Z.shape[0] - 2))
        nc = int(np.clip(np.searchsorted(cum, 0.90) + 1, 2, hi))
        self.pca = PCA(n_components=nc, random_state=self.seed).fit(Z)
        P = self.pca.transform(Z)
        try:
            self.cov = MinCovDet(support_fraction=0.8, random_state=self.seed).fit(P)
        except Exception:
            self.cov = EmpiricalCovariance().fit(P)

        d_sub = np.sqrt(np.maximum(0.0, self.cov.mahalanobis(P)))
        d_res = np.sqrt(np.mean((Z - self.pca.inverse_transform(P)) ** 2, axis=1))
        self.sub_med = float(np.median(d_sub))
        self.sub_iqr = max(1e-3, float(np.percentile(d_sub, 75) - np.percentile(d_sub, 25)))
        self.res_med = float(np.median(d_res))
        self.res_iqr = max(1e-3, float(np.percentile(d_res, 75) - np.percentile(d_res, 25)))
        return self

    def score_with_breakdown(self, X: np.ndarray) -> dict[str, np.ndarray]:
        X = np.asarray(X, dtype=np.float64)
        Xi = np.where(np.isfinite(X), X, self.impute)
        Z = np.clip((Xi - self.center) / self.scale, -6.0, 6.0)
        P = self.pca.transform(Z)
        d_sub_raw = np.sqrt(np.maximum(0.0, self.cov.mahalanobis(P)))
        d_res_raw = np.sqrt(np.mean((Z - self.pca.inverse_transform(P)) ** 2, axis=1))
        z_sub = np.maximum(0.0, (d_sub_raw - self.sub_med) / self.sub_iqr)
        z_res = np.maximum(0.0, (d_res_raw - self.res_med) / self.res_iqr)
        d_comb = np.sqrt(z_sub ** 2 + self.spe_weight * (z_res ** 2))
        return {
            "d_comb": d_comb,
            "z_subspace": z_sub,
            "z_residual_spe": z_res,
            "d_sub_raw": d_sub_raw,
            "d_res_raw": d_res_raw,
            "z_features": Z,
        }


def add_lr_from_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in LR_BASE:
        if c in out.columns:
            out[f"{c}_lr_abs"] = np.nan
    if {"subject", "task", "hand"}.issubset(out.columns):
        for _, g in out.groupby(["subject", "task"]):
            L, R = g[g["hand"] == "left"], g[g["hand"] == "right"]
            if len(L) and len(R):
                for c in LR_BASE:
                    if c in out.columns:
                        a, b = float(L[c].iloc[0]), float(R[c].iloc[0])
                        out.loc[g.index, f"{c}_lr_abs"] = abs(a - b) if np.isfinite(a) and np.isfinite(b) else np.nan
    return out


def _resolve_default_paths() -> tuple[Path, Path, Path]:
    for root in [
        Path("/kaggle/working/thermhand_out/final_v6"),
        Path("/kaggle/working/thermhand_out/final_v5"),
        Path("."),
    ]:
        c_pkl = root / "features_cache.pkl"
        b_json = root / "final_v6_bundle.json"
        if not b_json.exists():
            b_json = root / "final_v5_bundle.json"
        if c_pkl.exists() and b_json.exists():
            out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
            return c_pkl, b_json, out_dir / "model_bundle_excl_S17.json"
    out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
    return (
        Path("/kaggle/working/thermhand_out/final_v6/features_cache.pkl"),
        Path("/kaggle/working/thermhand_out/final_v6/final_v6_bundle.json"),
        out_dir / "model_bundle_excl_S17.json",
    )


def main() -> None:
    import argparse

    def_cache, def_bundle, def_out = _resolve_default_paths()
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache-pkl", default=str(def_cache))
    ap.add_argument("--bundle-json", default=str(def_bundle))
    ap.add_argument("--heldout-subject", default="S17")
    ap.add_argument("--out-json", default=str(def_out))
    args, _ = ap.parse_known_args()

    cache_path = Path(args.cache_pkl)
    bundle_path = Path(args.bundle_json)
    if not cache_path.exists():
        raise FileNotFoundError(f"Cache file not found: {cache_path}")
    if not bundle_path.exists():
        raise FileNotFoundError(f"Bundle JSON not found: {bundle_path}")

    blob = pickle.loads(cache_path.read_bytes())
    old_bundle = json.loads(bundle_path.read_text(encoding="utf-8"))
    rows = blob["rows"]

    df = pd.DataFrame([
        {
            "rec_id": r["rec_id"],
            "subject": r["subject"],
            "hand": r["hand"],
            "task": r["task"],
            **r["quality"],
            **r["variants"]["clean_+0.0C"],
        }
        for r in rows
    ])
    gate_ok = (
        (df["n_used"] >= 3)
        & (df["q_mask_ok_frac"] >= 0.85)
        & (df["q_area_cv"] <= 0.35)
        & (df["q_valid_joint_frac"] >= 0.50)
    )
    dfc = add_lr_from_df(df)
    cols = [c for c in old_bundle["feature_columns"] if c in dfc.columns]

    train_mask = (df["subject"] != args.heldout_subject) & gate_ok
    scorer = CohortScorer(seed=42, spe_weight=0.5).fit(dfc.loc[train_mask, cols].to_numpy(dtype=np.float64))
    train_scores = scorer.score_with_breakdown(dfc.loc[train_mask, cols].to_numpy(dtype=np.float64))["d_comb"]
    cut_p95 = float(np.percentile(train_scores, 95))

    reg_path = Path("/kaggle/working/thermhand_out/final_v6") / "registration.json"
    if not reg_path.exists():
        raise SystemExit(f"{reg_path} missing: run cell 2 main first.")
    reg_info = json.loads(reg_path.read_text(encoding="utf-8"))
    if not reg_info.get("joints_ok", False):
        print("WARNING: registration gate failed; joints disabled in this bundle.")

    export = {
        "schema_version": "6.1.0-single-subject-holdout",
        "heldout_subject": args.heldout_subject,
        "train_subjects": sorted(df.loc[train_mask, "subject"].unique().tolist()),
        "n_train_recordings": int(train_mask.sum()),
        "feature_columns": cols,
        "impute": scorer.impute.tolist(),
        "center": scorer.center.tolist(),
        "scale": scorer.scale.tolist(),
        "clip_z": 6.0,
        "pca_mean": scorer.pca.mean_.tolist(),
        "pca_components": scorer.pca.components_.tolist(),
        "mcd_location": scorer.cov.location_.tolist(),
        "mcd_precision": scorer.cov.precision_.tolist(),
        "sub_med": scorer.sub_med,
        "sub_iqr": scorer.sub_iqr,
        "res_med": scorer.res_med,
        "res_iqr": scorer.res_iqr,
        "spe_weight": scorer.spe_weight,
        "d_cut_p95": cut_p95,
        "quality_gate": {
            "min_n_used": 3,
            "min_mask_ok_frac": 0.85,
            "max_area_cv": 0.35,
            "min_valid_joint_frac": 0.50,
        },
        "thermal_orient": reg_info["orient"],
        "rgb_rotation_k": int(old_bundle.get("rgb_rotation_k", 1)),
        "M_global": reg_info["M"],
        "canonical_hw": [480, 640],
        "disclaimer": DISCLAIMER,
    }
    out_path = Path(args.out_json)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(export, indent=2), encoding="utf-8")
    print(
        f"Saved {out_path} trained on {len(export['train_subjects'])} subjects "
        f"({export['n_train_recordings']} recs), cut_p95={cut_p95:.4f}"
    )


if __name__ == "__main__":
    main()

Saved /kaggle/working/model_bundle_excl_S17.json trained on 16 subjects (62 recs), cut_p95=4.0777


In [4]:
import importlib, sys, os, urllib.request
print("python", sys.version.split()[0])
try:
    import mediapipe as mp
    print("mediapipe", mp.__version__, "| has solutions:", hasattr(mp, "solutions"))
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision as mp_vision
    print("tasks API import: OK")
except Exception as e:
    print("IMPORT FAILED:", repr(e))

task = "/tmp/hand_landmarker.task"
print("task file exists:", os.path.exists(task),
      os.path.getsize(task) if os.path.exists(task) else "")
try:
    urllib.request.urlopen("https://storage.googleapis.com", timeout=5)
    print("internet: OK")
except Exception as e:
    print("internet: BLOCKED ->", repr(e))

python 3.13.16
mediapipe 1.2.3 | has solutions: False
tasks API import: OK
task file exists: True 7819105
internet: BLOCKED -> <HTTPError 400: 'Bad Request'>


In [5]:
import glob
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

TASK = "/tmp/hand_landmarker.task"
opts = mp_vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=TASK),
    running_mode=mp_vision.RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3,
)
try:
    det = mp_vision.HandLandmarker.create_from_options(opts)
    print("create: OK")
except Exception as e:
    print("CREATE FAILED:", repr(e))
    raise

# 1) Dummy frame: checks the runtime works at all
try:
    r = det.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.zeros((480, 640, 3), np.uint8)))
    print("detect on blank frame: OK")
except Exception as e:
    print("DETECT FAILED:", repr(e))

# 2) Real RGB frame from the dataset, with the same rotation as the pipeline
vids = [p for p in glob.glob("/kaggle/input/**/*.mp4", recursive=True)
        if "thermal" not in p.lower() and "radiometric" not in p.lower()]
print("rgb videos found:", len(vids))
if vids:
    cap = cv2.VideoCapture(vids[0])
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, n // 2)
    ok, bgr = cap.read()
    cap.release()
    print("frame read:", ok, "| shape:", None if not ok else bgr.shape)
    if ok:
        rgb = np.ascontiguousarray(np.rot90(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), 1))
        for k in range(4):
            img = np.ascontiguousarray(np.rot90(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), k))
            try:
                r = det.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=img))
                print(f"rot k={k}: hands={len(r.hand_landmarks)}")
            except Exception as e:
                print(f"rot k={k}: DETECT FAILED {e!r}")
det.close()

create: OK
detect on blank frame: OK
rgb videos found: 0


INFO: [environment.cc:36] Creating LiteRT environment with options
INFO: [accelerator_registry.cc:54] RegisterAccelerator: ptr=0x785ee80332b0, name=CpuAccelerator
INFO: [cpu_registry.cc:75] XNNPACK CPU accelerator registered.
INFO: [compiled_model.cc:1022] Flatbuffer model initialized directly from incoming litert model.
INFO: [environment.cc:36] Creating LiteRT environment with options
INFO: [accelerator_registry.cc:54] RegisterAccelerator: ptr=0x785efc0057e0, name=CpuAccelerator
INFO: [cpu_registry.cc:75] XNNPACK CPU accelerator registered.
INFO: [compiled_model.cc:1022] Flatbuffer model initialized directly from incoming litert model.
INFO: [accelerator_registry.cc:43] DestroyAccelerator: ptr=0x785ee80332b0, name=CpuAccelerator
INFO: [accelerator_registry.cc:43] DestroyAccelerator: ptr=0x785efc0057e0, name=CpuAccelerator


In [6]:
import glob, numpy as np, cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

ROOT = "/kaggle/input/datasets/podakantisatyajith/thermal-data-collection/FLIROne"
vids = [p for p in glob.glob(f"{ROOT}/**/*.mp4", recursive=True)
        if "thermal" not in p.rsplit("/", 1)[-1].lower()
        and "radiometric" not in p.rsplit("/", 1)[-1].lower()]
print("rgb videos:", len(vids))
print("sample:", vids[:2])

det = mp_vision.HandLandmarker.create_from_options(mp_vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="/tmp/hand_landmarker.task"),
    running_mode=mp_vision.RunningMode.IMAGE, num_hands=1,
    min_hand_detection_confidence=0.3, min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3))

cap = cv2.VideoCapture(vids[0])
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
found = {k: 0 for k in range(4)}
for fr in np.linspace(0, n - 1, 6).astype(int):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(fr))
    ok, bgr = cap.read()
    if not ok:
        continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    for k in range(4):
        img = np.ascontiguousarray(np.rot90(rgb, k))
        r = det.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=img))
        found[k] += int(bool(r.hand_landmarks))
cap.release(); det.close()
print("hands detected per rotation k (out of 6 frames):", found)

INFO: [environment.cc:36] Creating LiteRT environment with options
INFO: [accelerator_registry.cc:54] RegisterAccelerator: ptr=0x785ef4001e30, name=CpuAccelerator
INFO: [cpu_registry.cc:75] XNNPACK CPU accelerator registered.
INFO: [compiled_model.cc:1022] Flatbuffer model initialized directly from incoming litert model.


rgb videos: 68
sample: ['/kaggle/input/datasets/podakantisatyajith/thermal-data-collection/FLIROne/S1/left hand/open_close_RGB.mp4', '/kaggle/input/datasets/podakantisatyajith/thermal-data-collection/FLIROne/S1/left hand/static_RGB.mp4']


INFO: [environment.cc:36] Creating LiteRT environment with options
INFO: [accelerator_registry.cc:54] RegisterAccelerator: ptr=0x785ef4073150, name=CpuAccelerator
INFO: [cpu_registry.cc:75] XNNPACK CPU accelerator registered.
INFO: [compiled_model.cc:1022] Flatbuffer model initialized directly from incoming litert model.


hands detected per rotation k (out of 6 frames): {0: 6, 1: 6, 2: 6, 3: 6}


INFO: [accelerator_registry.cc:43] DestroyAccelerator: ptr=0x785ef4001e30, name=CpuAccelerator
INFO: [accelerator_registry.cc:43] DestroyAccelerator: ptr=0x785ef4073150, name=CpuAccelerator
